[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvvvvivekkk/AI-Engineer/blob/main/00-setup/notebooks/02-env-files-safely.ipynb)

# 00 · Notebook 2 — Working with `.env` files safely

**Goal:** learn the safe pattern for API keys that every later module uses.

Words used here:
- **API key** — a secret password that lets your code use a paid online service. Anyone with it can spend your money.
- **Environment variable** — a named value stored outside your code, in the settings of the running program.
- **`.env` file** — a text file of `NAME=value` lines that holds your secrets locally. It is never committed to Git.
- **python-dotenv** — a small package that reads `.env` into environment variables.

This notebook never uses a real key. It creates a throw-away `.env` with a fake value in a temporary folder and deletes it at the end.

## 1. Install check

`python-dotenv` is installed under the name `python-dotenv` but imported as `dotenv`. In Colab it may be missing; the cell installs it quietly if needed using `%pip` (the notebook version of `pip`, which installs into the running kernel).

In [1]:
try:
    import dotenv
except ImportError:
    %pip install -q python-dotenv
    import dotenv

import importlib.metadata as meta
print("python-dotenv version:", meta.version("python-dotenv"))

python-dotenv version: 1.2.3


## 2. Environment variables with `os`

`os.environ` behaves like a dictionary of every environment variable. `os.getenv("NAME")` returns the value, or `None` if it does not exist. Asking for a missing key with `getenv` never crashes, which makes it the safe way to read keys.

In [2]:
import os

print("HOME or USERPROFILE is set:", bool(os.getenv("HOME") or os.getenv("USERPROFILE")))
print("A made-up variable:", os.getenv("DEFINITELY_NOT_SET_123"))
print("With a default value:", os.getenv("DEFINITELY_NOT_SET_123", "fallback"))

HOME or USERPROFILE is set: True
A made-up variable: None
With a default value: fallback


## 3. Make a practice `.env` file

We write a `.env` into a **temporary folder** (a folder the operating system cleans up), so we cannot accidentally commit it. The value is obviously fake.

In [3]:
import tempfile
from pathlib import Path

practice_dir = Path(tempfile.mkdtemp())
env_path = practice_dir / ".env"
env_path.write_text(
    "# practice file - fake values only\n"
    "DEMO_API_KEY=demo-not-a-real-key-123\n"
    "APP_ENV=development\n",
    encoding="utf-8",
)
print("Wrote:", env_path)
print("File contents (safe to show - fake values):")
print(env_path.read_text())

Wrote: /tmp/tmp5v0sgo_v/.env
File contents (safe to show - fake values):
# practice file - fake values only
DEMO_API_KEY=demo-not-a-real-key-123
APP_ENV=development



## 4. Load it with `load_dotenv`

`load_dotenv(path)` copies each `NAME=value` line into the environment variables of this running Python. It returns `True` if it found the file.

Then we read the value with `os.getenv`. Notice how we **report** that the key is set without showing it.

In [4]:
from dotenv import load_dotenv

found = load_dotenv(env_path)
print("File found and loaded:", found)

key = os.getenv("DEMO_API_KEY")
print("DEMO_API_KEY is set:", key is not None)
print("APP_ENV =", os.getenv("APP_ENV"))   # not a secret, fine to print

File found and loaded: True
DEMO_API_KEY is set: True
APP_ENV = development


## 5. Showing a key safely (when you must)

Sometimes you need to confirm *which* key is loaded. Show only that it exists and its length, or at most the last 2 characters. Never the whole thing. The function below does that.

In [5]:
def describe_secret(value):
    """Describe a secret without revealing it."""
    if value is None:
        return "NOT SET"
    if value.strip() == "" or "your-key-here" in value:
        return "EMPTY or PLACEHOLDER"
    return f"set ({len(value)} characters, ends with ...{value[-2:]})"

print("DEMO_API_KEY:", describe_secret(os.getenv("DEMO_API_KEY")))
print("OPENAI_API_KEY:", describe_secret(os.getenv("OPENAI_API_KEY")))

DEMO_API_KEY: set (23 characters, ends with ...23)
OPENAI_API_KEY: NOT SET


## 6. `override`: who wins if a variable is already set?

By default `load_dotenv` does **not** replace a variable that already exists in the environment. That lets a server's real settings win over a developer's `.env`. Pass `override=True` to let the file win.

In [6]:
os.environ["APP_ENV"] = "production"          # pretend the server set this

load_dotenv(env_path)                            # default: override=False
print("After default load :", os.getenv("APP_ENV"))

load_dotenv(env_path, override=True)
print("After override=True:", os.getenv("APP_ENV"))

After default load : production
After override=True: development


## 7. Read a `.env` without touching the environment

`dotenv_values(path)` returns the file as a plain dictionary. Useful for checking which names a file defines. We print only the **names**, not the values.

In [7]:
from dotenv import dotenv_values

values = dotenv_values(env_path)
print("Names defined in the file:", list(values.keys()))

Names defined in the file: ['DEMO_API_KEY', 'APP_ENV']


## 8. Is `.env` protected by `.gitignore`?

A **`.gitignore`** file lists files Git must never track. This small helper reads a `.gitignore` and checks whether `.env` is covered. We test it on two pretend files: a good one and a dangerous one. (Real Git matching has more rules; this is a simple check.)

In [8]:
def env_is_ignored(gitignore_text):
    lines = [line.strip() for line in gitignore_text.splitlines()]
    lines = [line for line in lines if line and not line.startswith("#")]
    return ".env" in lines or "*.env" in lines or ".env*" in lines

good = "# secrets\n.env\n.venv/\n__pycache__/\n"
bad = ".venv/\n__pycache__/\n"
print("Good .gitignore protects .env:", env_is_ignored(good))
print("Bad  .gitignore protects .env:", env_is_ignored(bad))

repo_gitignore = Path("../../.gitignore")
if repo_gitignore.exists():
    print("This course repo protects .env:", env_is_ignored(repo_gitignore.read_text()))

Good .gitignore protects .env: True
Bad  .gitignore protects .env: False
This course repo protects .env: True


## 9. The pattern every later module uses

1. Try to load the key (from `.env` locally, or Colab Secrets on Colab).
2. If it exists, make the real API call.
3. If not, use a clearly-labelled **offline stand-in** so the notebook still runs.

`google.colab.userdata.get(...)` reads a value from Colab's **Secrets** panel (the key icon in Colab's left sidebar). Outside Colab that import fails, so we guard it.

In [9]:
def get_key(name):
    # 1) Colab Secrets panel (only exists in Colab)
    try:
        from google.colab import userdata
        try:
            return userdata.get(name)
        except Exception:
            pass  # secret not created, or notebook access not switched on
    except ImportError:
        pass
    # 2) Environment variables, filled from .env by load_dotenv()
    load_dotenv()
    value = os.getenv(name)
    if value and "your-key-here" not in value:
        return value
    return None


def ask_llm(question):
    key = get_key("OPENAI_API_KEY")
    if key:
        # The real call is taught in module 07, for example:
        # from openai import OpenAI
        # client = OpenAI(api_key=key)
        return "(real key found - module 07 makes the real call here)"
    return f"[OFFLINE STAND-IN] I would answer: {question!r}"


print(ask_llm("Why should .env never be committed?"))

[OFFLINE STAND-IN] I would answer: 'Why should .env never be committed?'


## 10. Clean up

We delete the practice file and folder, and remove the demo variables from this session.

In [10]:
import shutil

shutil.rmtree(practice_dir)
for name in ["DEMO_API_KEY", "APP_ENV"]:
    os.environ.pop(name, None)
print("Practice folder removed:", not practice_dir.exists())
print("DEMO_API_KEY now:", os.getenv("DEMO_API_KEY"))

Practice folder removed: True
DEMO_API_KEY now: None


## Summary

- Read keys with `os.getenv("NAME")` — it returns `None` instead of crashing.
- Keep keys in `.env`, load them with `load_dotenv()`, and list `.env` in `.gitignore`.
- Commit a `.env.example` with placeholder values so teammates know which names to set.
- In Colab, use the **Secrets** panel and `userdata.get`.
- Never print a key. Notebook outputs are saved in the file.
- If a key ever leaks: **revoke it on the provider's website first**, then clean up.